In [0]:
%sql
-- 1. Aseguramos que la tabla no tenga basura previa
-- DROP TABLE IF EXISTS products.bronze_scraped_products_with_json;

In [0]:
%sql
    
-- 1. Crear la tabla SOLO si no existe
-- Usamos TBLPROPERTIES para soportar nombres de columnas con espacios (como tus 'Opcion 1')
CREATE TABLE IF NOT EXISTS workspace.products.bronze_scraped_products_with_json (
  scraped_at TIMESTAMP,
  product_id STRING,
  retailer STRING,
  raw_data STRING,
  file_metadata_path STRING
)
USING DELTA
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
);

In [0]:
%sql
INSERT INTO workspace.products.bronze_scraped_products_with_json
SELECT 
  to_timestamp(scraped_at) AS scraped_at,
  product_id,
  retailer,
  to_json(raw_data) AS raw_data,
  _metadata.file_path AS file_metadata_path
FROM read_files(
  '/Volumes/workspace/products/products_tracker/scraped',
  format => 'json',
  multiLine => true
);

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.products.bronze_scraped_products_clean AS
WITH clean_bronze_products AS (
    SELECT
        scraped_at ,
        product_id,
        retailer,
        get_json_object(raw_data, '$.sku') AS sku,
        get_json_object(raw_data, '$.name') AS name,
        get_json_object(raw_data, '$.brand') AS brand,
        get_json_object(raw_data, '$.main_category') AS main_category,
        get_json_object(raw_data, '$.sub_category') AS sub_category,
        get_json_object(raw_data, '$.list_price') AS list_price,
        get_json_object(raw_data, '$.cash_price') AS cash_price,
        get_json_object(raw_data, '$.stock') AS is_in_stock,
        get_json_object(raw_data, '$.installments') AS installments,
        get_json_object(raw_data, '$.description') AS description,
        get_json_object(raw_data, '$.specifications') AS specifications,
        get_json_object(raw_data, '$.rating') AS rating
    FROM
        products.bronze_scraped_products_with_json
),
deduplicated_bronze_products AS (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY product_id, retailer, scraped_at
           ORDER BY scraped_at DESC
         ) as rn
  FROM clean_bronze_products
)
SELECT *
FROM deduplicated_bronze_products
WHERE rn = 1;